# Stage 5 — Retriever pipeline, latency & accuracy benchmarking

Input: the three Milvus collections from `4_milvus_indexing.ipynb` (`chunks_flat`, `chunks_hnsw`, `chunks_ivf_flat`), plus `data/chunks/chunks.jsonl` for sampling.

This notebook:

1. Builds a small synthetic query set with known ground truth — sample real chunks, ask GPT-4o to write a question each one answers, so the "correct" `chunk_id` is known for every query
2. Builds a retriever pipeline: embed a query once, search all three collections with that same vector
3. Times just the Milvus `.search()` call per index (embedding time is shared overhead, excluded from the index comparison)
4. Computes **Recall@k** and **average top-1 similarity score** per index — the accuracy side
5. Computes **mean / p50 / p95 latency** per index — the speed side
6. Saves raw per-query results to `data/logs/stage5_benchmark.jsonl`

Requires Milvus running (`docker compose up -d`, already up from Stage 4) and the same Azure OpenAI `.env` vars as Stage 3.


In [1]:
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI
from pymilvus import connections, Collection
from tqdm.auto import tqdm

/Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CHUNKS_PATH = DATA_DIR / "chunks" / "chunks.jsonl"
QUESTION_CACHE_DIR = DATA_DIR / "eval_cache"
LOGS_DIR = DATA_DIR / "logs"

for d in (QUESTION_CACHE_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
AZURE_OPENAI_GPT4O_DEPLOYMENT = os.environ["AZURE_OPENAI_GPT4O_DEPLOYMENT"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

MILVUS_HOST = "localhost"
MILVUS_PORT = "19530"
connections.connect(alias="default", host=MILVUS_HOST, port=MILVUS_PORT)

TOP_K = 5

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = [json.loads(line) for line in f]

print(f"loaded {len(all_chunks)} chunks")

loaded 1301 chunks


/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_34047/3839770582.py:26: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(alias="default", host=MILVUS_HOST, port=MILVUS_PORT)


## Load the three collections

Same `INDEX_CONFIGS` shape as Stage 4, but only the `metric_type`/`search_params` half matters here — the indexes themselves already exist. `.load()` is idempotent, safe to call again even if Milvus already has these loaded from Stage 4.


In [3]:
INDEX_CONFIGS = {
    "flat": {"metric_type": "COSINE", "search_params": {}},
    "hnsw": {"metric_type": "COSINE", "search_params": {"ef": 64}},
    "ivf_flat": {"metric_type": "COSINE", "search_params": {"nprobe": 10}},
}

collections = {key: Collection(f"chunks_{key}") for key in INDEX_CONFIGS}
for collection in collections.values():
    collection.load()

print("collections ready:", {key: c.num_entities for key, c in collections.items()})

collections ready: {'flat': 1301, 'hnsw': 1301, 'ivf_flat': 1301}


/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_34047/296114827.py:7: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collections = {key: Collection(f"chunks_{key}") for key in INDEX_CONFIGS}
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_34047/296114827.py:9: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_34047/296114827.py:11: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print("collections ready:", {key: c.num_entities for key, c in collections.items()})


## Build a synthetic query set with known ground truth

Sample real chunks — deliberately across **all three chunk types**, not just text, since this is a multi-modal RAG and the eval set should actually exercise table/image retrieval too, not silently skip them. Each sampled chunk's `chunk_id` becomes the known-correct answer for whatever question gets generated from it.


In [4]:
random.seed(42)


def sample_chunks_by_type(chunks: list[dict], chunk_type: str, n: int) -> list[dict]:
    pool = [c for c in chunks if c["chunk_type"] == chunk_type]
    return random.sample(pool, min(n, len(pool)))


sampled_chunks = (
    sample_chunks_by_type(all_chunks, "text", 12)
    + sample_chunks_by_type(all_chunks, "table", 4)
    + sample_chunks_by_type(all_chunks, "image", 4)
)

print(f"sampled {len(sampled_chunks)} chunks for the eval set")
pd.DataFrame(sampled_chunks)["chunk_type"].value_counts()

sampled 20 chunks for the eval set


chunk_type
text     12
table     4
image     4
Name: count, dtype: int64

## Generate one question per sampled chunk (GPT-4o)

Cached by `chunk_id` (exact-match caching, same pattern as Stage 3's `caption_image_cached`) — re-running this notebook shouldn't regenerate questions that already exist. Content is truncated to 2000 characters before prompting, mainly a safety margin against an unusually long table's Markdown.


In [5]:
QUESTION_PROMPT = (
    "You are creating a test question for a search system, based on the passage below. "
    "Write ONE specific question that this passage directly and uniquely answers. "
    "Do not reference 'the passage' or 'the text' in the question — ask it as a standalone question. "
    "Return only the question, nothing else.\n\nPassage:\n{content}"
)


def generate_question_cached(chunk: dict) -> str:
    cache_path = QUESTION_CACHE_DIR / f"{chunk['chunk_id']}.txt"
    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8")

    response = client.chat.completions.create(
        model=AZURE_OPENAI_GPT4O_DEPLOYMENT,
        messages=[{"role": "user", "content": QUESTION_PROMPT.format(content=chunk["content"][:2000])}],
        max_tokens=100,
    )
    question = response.choices[0].message.content.strip()

    cache_path.write_text(question, encoding="utf-8")
    return question

In [6]:
eval_set = []

for chunk in tqdm(sampled_chunks, desc="generating questions"):
    question = generate_question_cached(chunk)
    eval_set.append({
        "question": question,
        "expected_chunk_id": chunk["chunk_id"],
        "expected_chunk_type": chunk["chunk_type"],
        "expected_doc_id": chunk["doc_id"],
        "expected_page_number": chunk["page_number"],
    })

pd.DataFrame(eval_set)

generating questions: 100%|██████████| 20/20 [00:00<00:00, 6747.59it/s]


,question,expected_chunk_id,expected_chunk_type,expected_doc_id,expected_page_number
0,What demonstrates more variation in estimates ...,271c6890dc2a430ebf22ae950ab009f3,text,IPCC_AR6_WGI_Chapter07.pdf,80
1,What advantage does the radiative kernel appro...,6d8d07b79d8b406cbd564d578ccd3ccb,text,IPCC_AR6_WGI_Chapter07.pdf,20
2,What factors contribute to permafrost degradat...,028ca87fa9374bab8127d7ec83030a96,text,IPCC_AR6_WGI_Chapter08.pdf,18
3,What is the title of the study published by Po...,dc592958e93b4406881c295369e45fc6,text,IPCC_AR6_WGI_Chapter07.pdf,130
4,What is the title and publication year of the ...,3b9e59df2c784b75a8b7e27150ce6a81,text,IPCC_AR6_WGI_Chapter07.pdf,124
5,How do aerosols released by human activities i...,dbeb7523ea8744e3918d230b0f0c10e5,text,IPCC_AR6_WGI_Chapter07.pdf,100
6,What evidence suggests that the true equilibri...,eafa938cb2b846b29ec4e5d59278aad1,text,IPCC_AR6_WGI_Chapter07.pdf,74
7,What is the journal title and volume number in...,f0480d0c42764a76b48dbc1eaef9c5a2,text,IPCC_AR6_WGI_Chapter08.pdf,145
8,What effect could freshwater input from meltin...,0b2478fcfba34aad945e25679cdb5c90,text,IPCC_AR6_WGI_Chapter07.pdf,63
9,What study analyzed the amplification of the N...,e412caa306914cabba9a44e0b7ca96eb,text,IPCC_AR6_WGI_Chapter08.pdf,113


## Retriever pipeline

`embed_query` calls the raw `openai` client directly (not the LangChain wrapper from Stage 3) since this notebook only ever embeds one query string at a time — no batching to justify the extra dependency.

`search_collection` times **only** the Milvus `.search()` call. `retrieve_all_indexes` embeds a query exactly once, then reuses that same vector across all three collections — this is what makes the latency comparison between indexes fair, since the (shared, identical) embedding cost is excluded from each index's own timing.


In [7]:
def embed_query(text: str) -> list[float]:
    response = client.embeddings.create(model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT, input=text)
    return response.data[0].embedding


def search_collection(collection: Collection, query_vector: list[float], config: dict, top_k: int):
    start = time.perf_counter()
    results = collection.search(
        data=[query_vector],
        anns_field="embedding",
        param={"metric_type": config["metric_type"], "params": config["search_params"]},
        limit=top_k,
        output_fields=["chunk_id", "doc_id", "page_number", "chunk_type"],
    )[0]
    latency = time.perf_counter() - start
    return results, latency


def retrieve_all_indexes(query_text: str, top_k: int = TOP_K) -> dict:
    query_vector = embed_query(query_text)

    per_index = {}
    for key, collection in collections.items():
        hits, latency = search_collection(collection, query_vector, INDEX_CONFIGS[key], top_k)
        per_index[key] = {"hits": hits, "latency_sec": latency}
    return per_index

## Run the benchmark

One row per `(query, index)` pair — so 20 questions × 3 indexes = 60 rows. `hit_at_k` checks whether the known-correct `chunk_id` actually appears anywhere in that index's top-`TOP_K` results, which is exactly what Recall@k needs.


In [8]:
records = []

for item in tqdm(eval_set, desc="benchmarking queries"):
    per_index = retrieve_all_indexes(item["question"])

    for key, result in per_index.items():
        hits = result["hits"]
        hit_chunk_ids = [h.entity.get("chunk_id") for h in hits]
        top1 = hits[0] if hits else None

        records.append({
            "question": item["question"],
            "expected_chunk_id": item["expected_chunk_id"],
            "expected_chunk_type": item["expected_chunk_type"],
            "index": key,
            "latency_sec": result["latency_sec"],
            "top1_chunk_id": top1.entity.get("chunk_id") if top1 else None,
            "top1_score": top1.distance if top1 else None,
            "hit_at_k": item["expected_chunk_id"] in hit_chunk_ids,
        })

results_df = pd.DataFrame(records)
results_df.head(10)

benchmarking queries:   0%|          | 0/20 [00:00<?, ?it/s]/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_34047/1640012819.py:8: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_34047/1640012819.py:8: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(
benchmarking queries: 100%|██████████| 20/20 [00:04<00:00,  4.37it/s]


,question,expected_chunk_id,expected_chunk_type,index,latency_sec,top1_chunk_id,top1_score,hit_at_k
0,What demonstrates more variation in estimates ...,271c6890dc2a430ebf22ae950ab009f3,text,flat,0.144288,271c6890dc2a430ebf22ae950ab009f3,0.779315,True
1,What demonstrates more variation in estimates ...,271c6890dc2a430ebf22ae950ab009f3,text,hnsw,0.028491,271c6890dc2a430ebf22ae950ab009f3,0.779315,True
2,What demonstrates more variation in estimates ...,271c6890dc2a430ebf22ae950ab009f3,text,ivf_flat,0.018679,271c6890dc2a430ebf22ae950ab009f3,0.779315,True
3,What advantage does the radiative kernel appro...,6d8d07b79d8b406cbd564d578ccd3ccb,text,flat,0.007941,597e95f19e3b4f7191f4e168dd3f9172,0.562468,True
4,What advantage does the radiative kernel appro...,6d8d07b79d8b406cbd564d578ccd3ccb,text,hnsw,0.007844,597e95f19e3b4f7191f4e168dd3f9172,0.562467,True
5,What advantage does the radiative kernel appro...,6d8d07b79d8b406cbd564d578ccd3ccb,text,ivf_flat,0.005229,597e95f19e3b4f7191f4e168dd3f9172,0.562468,True
6,What factors contribute to permafrost degradat...,028ca87fa9374bab8127d7ec83030a96,text,flat,0.010746,a6bf13d194314bbc91f1067a3fd7f3ed,0.509080,True
7,What factors contribute to permafrost degradat...,028ca87fa9374bab8127d7ec83030a96,text,hnsw,0.009641,a6bf13d194314bbc91f1067a3fd7f3ed,0.509080,True
8,What factors contribute to permafrost degradat...,028ca87fa9374bab8127d7ec83030a96,text,ivf_flat,0.013469,a6bf13d194314bbc91f1067a3fd7f3ed,0.509080,True
9,What is the title of the study published by Po...,dc592958e93b4406881c295369e45fc6,text,flat,0.005148,dc592958e93b4406881c295369e45fc6,0.591482,True


## Aggregate: latency (mean/p50/p95) + accuracy (Recall@k, avg top-1 score)

This table is the actual deliverable — the direct comparison across FLAT / HNSW / IVF_FLAT on both speed and accuracy. p95 matters as much as the mean here: it shows the worst-case tail an index can produce, which a mean alone would hide.


In [9]:
summary = results_df.groupby("index").agg(
    mean_latency_sec=("latency_sec", "mean"),
    p50_latency_sec=("latency_sec", lambda s: np.percentile(s, 50)),
    p95_latency_sec=("latency_sec", lambda s: np.percentile(s, 95)),
    recall_at_k=("hit_at_k", "mean"),
    avg_top1_score=("top1_score", "mean"),
)

display(summary)

,mean_latency_sec,p50_latency_sec,p95_latency_sec,recall_at_k,avg_top1_score
index,,,,,
flat,0.014350,0.007010,0.021086,0.8,0.66757
hnsw,0.005656,0.003934,0.010584,0.8,0.66757
ivf_flat,0.005484,0.003507,0.013730,0.8,0.66757


In [10]:
def save_jsonl(units: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for unit in units:
            f.write(json.dumps(unit) + "\n")


save_jsonl(records, LOGS_DIR / "stage5_benchmark.jsonl")
print(f"saved {len(records)} benchmark records to {LOGS_DIR / 'stage5_benchmark.jsonl'}")

saved 60 benchmark records to /Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/data/logs/stage5_benchmark.jsonl


## Next: Stage 6 — MMR reranking

Raw benchmark results saved to `data/logs/stage5_benchmark.jsonl` (60 rows — one per question × index). Stage 6 applies MMR reranking on top of one chosen index's retrieved results, trading a little top-1 relevance for diversity across the retrieved chunks before they go into the final prompt.
